In [32]:
import pandas as pd
import numpy as np
import pickle
from lightgbm import LGBMRegressor
MODELS_CACHE = {}

In [33]:
# Read in full csv file
df = pd.read_csv("full_data.csv")

In [34]:
# Pitch types and sides
pitches = ("FA", "SL", "CH", "CU", "FC", "SI", "FS")
lr = ["L", "R"]

# Features & target
features = ['inducedVertBreak', 'horzBreak', 'extension', 'relX', 'relZ', 'releaseVelocity', 'spinRate']
target = ["rv"]

# Set test year
test = df[df["year"] == 2026]

In [35]:
# Filter test df so only pitch types we are looking for are included
test = test[test["pitchType"].isin(pitches)]

In [36]:
# Function that reads in a pitch, loads a pre-trained model, and predicts run value of that pitch
def pred_rv(test, features, target, pitch, side):
    if (pitch, side) not in MODELS_CACHE:
        with open(f'models/26_{pitch}_{side}.pkl', 'rb') as f:
            MODELS_CACHE[(pitch, side)] = pickle.load(f)

    model = MODELS_CACHE[(pitch, side)]
    X_test = test[features]
    predictions = model.predict(X_test) 
    
    return pd.Series(predictions, index=test.index)

In [37]:
rows = []
i=1

# Loop thru every pitch(row) of the test set and call pred_rv to predict each run value
for index, row in test.iterrows():
    pitch = row["pitchType"]
    side = row["pitcherHand"]

    test = pd.DataFrame([row])
    
    test["xrv"] = float(pred_rv(test, features, target, pitch, side).iloc[0]) 
    rows.append(test)

    # Print a number for every 100,000 pitches
    if i % 100000 == 0:
        print(i)

    i = i+1

100000
200000
300000
400000
500000
600000
700000
800000
900000
1000000
1100000
1200000
1300000
1400000
1500000
1600000
1700000


In [38]:
# Concatenate all pitches into one df
new = pd.concat(rows)

In [39]:
# Group by pitcherId, get mean of relevnt columns
stuff = new.groupby(["pitcherId", "pitchType"]).agg(
    Pitcher = ("Pitcher", "first"),
    year = ("year", "mean"),
    team = ("pitchingTeam", "first"),
    Hand = ("pitcherHand", "first"),
    Count = ("pitchType", "count"),
    RelSpeed = ("releaseVelocity", "mean"),
    IVB = ("inducedVertBreak", "mean"),
    HB = ("horzBreak", "mean"),
    SpinRate = ("spinRate", "mean"),
    RelSide = ("relX", "mean"),
    RelHeight = ("relZ", "mean"),
    Extension = ("extension", "mean"),
    xrv = ("xrv", "mean")
).reset_index()

In [40]:
stuff = stuff[["Pitcher", "pitcherId", "team", "pitchType", "Hand", "Count", "RelSpeed", "IVB", "HB", "SpinRate", "RelSide", "RelHeight", "Extension", "xrv"]]

# Pitcher must throw at least 25 of a pitch type to qualify
stuff = stuff[stuff["Count"] >= 25]

In [41]:
stuff = stuff.rename(columns={'pitchType': 'PitchType'})

In [42]:
stuff[stuff["team"] == "URI"].sort_values("xrv")

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv
12300,P. Aikens,1299756340,URI,SL,L,183,72.773770,5.278142,16.740984,2663.830601,-2.823497,4.681967,4.208197,0.048941
1837,E. Maloney,1085415168,URI,CH,R,105,75.727619,9.952381,9.387619,1476.933333,0.046667,6.949524,4.892381,0.058851
2871,A. Jones,1101042688,URI,CU,R,40,76.402500,-17.792500,-10.932500,2581.225000,1.025000,5.837500,5.312500,0.062518
18036,M. Santos,1487988943,URI,SL,R,198,81.849495,-0.354545,-16.391919,2649.939394,2.654545,5.026768,5.262626,0.063084
3230,J. Sabbath,1110853376,URI,SL,R,253,78.682213,4.809091,-11.677075,2367.371542,1.799209,5.684190,5.823715,0.064397
16040,J. Cullen,1358089441,URI,CU,R,63,76.393651,-13.360317,-14.069841,2764.238095,1.220635,6.190476,5.269841,0.067156
16044,J. Cullen,1358089441,URI,SL,R,102,83.207843,2.313725,-5.055882,2614.931373,1.492157,5.834314,5.527451,0.067753
4413,L. Lavigueur,1143021056,URI,SL,R,130,78.691538,-5.034615,-19.984615,2549.246154,1.500769,5.895385,5.288462,0.070204
14363,C. Maloney,1336247552,URI,SL,R,65,82.427692,-3.096923,-6.824615,2565.892308,2.306154,5.763077,5.330769,0.070570
10689,J. Kopetski,1269533184,URI,FC,L,66,76.672727,7.915152,3.574242,1709.590909,-2.051515,5.624242,6.143939,0.070924


In [43]:
# Convert xrv to an above-average stat, resulting in our Stuff+ values for each pitcher/pitch

from scipy import stats

stuff["Stuff+"] = 100 + (-10 * stuff.groupby("PitchType")["xrv"].transform(stats.zscore))
stuff["Stuff+"] = stuff["Stuff+"].round().astype(int)

In [44]:
stuff.sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26547,D. Volantis,1979617275,TEX,CU,L,383,81.376762,-12.064752,3.326110,2586.378590,-2.091384,6.800522,5.224021,0.004106,153
5097,G. Helsel,1162012672,XAV,FC,L,181,78.374033,7.903867,7.519890,2188.348066,-2.481768,6.627072,5.295028,-0.002987,151
3598,A. Buczkowski,1123936875,CIN,SL,R,332,81.022590,10.767169,-12.430723,2579.635542,5.368976,3.634639,5.293976,0.020201,146
12360,H. Dietz,1301090561,ARK,CU,L,136,80.165441,-14.566912,7.788235,2741.213235,-2.185294,6.729412,5.277941,0.018632,142
168,D. Hamilton,1008376863,CARK,SI,R,62,90.466129,17.624194,17.533871,2455.758065,2.495161,5.132258,5.620968,0.048104,140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2317,A. Lyon,1094692864,COFC,CH,R,34,73.467647,0.408824,7.158824,1797.205882,4.094118,3.120588,6.073529,0.163694,45
2782,T. Beard,1100603904,FSU,SL,L,149,84.998658,2.993960,-2.782550,1401.993289,-0.508054,6.687919,5.310067,0.148153,41
5643,J. Rosenblitt,1164786998,OAK,CU,R,45,68.164444,5.333333,-7.193333,1821.866667,0.546667,5.844444,5.582222,0.156444,39
10579,J. Douthat,1266710507,VMI,SI,L,35,81.891429,2.794286,-10.402857,2283.571429,-3.197143,3.400000,5.022857,0.221865,31


In [45]:
# Best pitches in country
stuff.sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26547,D. Volantis,1979617275,TEX,CU,L,383,81.376762,-12.064752,3.326110,2586.378590,-2.091384,6.800522,5.224021,0.004106,153
5097,G. Helsel,1162012672,XAV,FC,L,181,78.374033,7.903867,7.519890,2188.348066,-2.481768,6.627072,5.295028,-0.002987,151
3598,A. Buczkowski,1123936875,CIN,SL,R,332,81.022590,10.767169,-12.430723,2579.635542,5.368976,3.634639,5.293976,0.020201,146
12360,H. Dietz,1301090561,ARK,CU,L,136,80.165441,-14.566912,7.788235,2741.213235,-2.185294,6.729412,5.277941,0.018632,142
168,D. Hamilton,1008376863,CARK,SI,R,62,90.466129,17.624194,17.533871,2455.758065,2.495161,5.132258,5.620968,0.048104,140
1170,M. Harsch,1065101304,WAKE,SI,R,65,96.141538,7.783077,15.910769,1816.892308,0.967692,5.933846,5.929231,0.049210,139
21330,B. Trosclair,1674977859,NWST,CU,L,29,78.096552,-3.244828,10.444828,2977.862069,-2.155172,5.551724,6.468966,0.023093,139
10385,R. Hughes,1262528256,TOL,SI,R,42,92.028571,14.850000,12.652381,2022.738095,1.614286,7.754762,5.790476,0.050860,138
12743,B. Purcell,1307080960,FSU,CH,R,93,85.370968,4.695699,15.644086,1378.559140,0.355914,6.738710,5.109677,0.042274,138
7816,D. Jackson,1205651242,UCSB,CH,L,148,76.597973,3.318919,-20.152027,2558.668919,-1.581081,5.831081,5.795270,0.042210,138


In [46]:
stuff[stuff["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
1837,E. Maloney,1085415168,URI,CH,R,105,75.727619,9.952381,9.387619,1476.933333,0.046667,6.949524,4.892381,0.058851,125
12300,P. Aikens,1299756340,URI,SL,L,183,72.773770,5.278142,16.740984,2663.830601,-2.823497,4.681967,4.208197,0.048941,123
16041,J. Cullen,1358089441,URI,FA,R,341,91.593548,21.456598,10.419648,2257.530792,1.110264,6.104985,6.004106,0.073858,119
7679,C. Grotyohann,1203468032,URI,FA,R,173,90.732370,20.983815,8.970520,2186.439306,1.881503,6.187861,5.882081,0.080201,116
18034,M. Santos,1487988943,URI,FA,R,192,92.588021,16.129167,7.259375,2465.114583,2.401042,5.323437,5.613542,0.084542,113
18036,M. Santos,1487988943,URI,SL,R,198,81.849495,-0.354545,-16.391919,2649.939394,2.654545,5.026768,5.262626,0.063084,111
14360,C. Maloney,1336247552,URI,FA,R,110,90.299091,19.080909,10.143636,2198.209091,1.944545,5.961818,5.870909,0.090212,110
2872,A. Jones,1101042688,URI,FA,R,143,89.686014,21.048252,7.764336,2156.853147,1.054545,5.839860,5.832867,0.091132,110
3230,J. Sabbath,1110853376,URI,SL,R,253,78.682213,4.809091,-11.677075,2367.371542,1.799209,5.684190,5.823715,0.064397,110
2871,A. Jones,1101042688,URI,CU,R,40,76.402500,-17.792500,-10.932500,2581.225000,1.025000,5.837500,5.312500,0.062518,109


In [1]:
# Exploring the best of each pitch type nation wide

In [47]:
stuff[stuff["PitchType"] == "FA"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
5073,J. Music,1161348114,CAMP,FA,R,212,92.099528,21.347170,13.038208,2209.424528,3.513208,5.965566,5.712736,0.047375,133
4975,B. Frederick,1158279680,TENN,FA,R,319,82.363636,1.687461,7.452351,2176.504702,2.923197,1.651724,5.718182,0.047789,132
7239,R. Marohn,1197348608,NCST,FA,L,381,91.790289,17.194226,-13.963780,2262.847769,-3.645144,5.567454,6.154068,0.049415,132
20160,S. Garcia,1602557566,LSU,FA,L,157,92.716561,21.284076,-10.994904,2441.662420,-1.621656,6.249045,6.138854,0.049356,132
24696,J. Barberi,1862463014,FLA,FA,R,260,97.502308,20.915385,10.915385,2533.519231,1.840769,6.333077,5.432308,0.051253,131
7286,S. Matson,1198395392,USC,FA,L,383,90.302611,16.579112,-15.990862,2311.707572,-1.660836,4.800783,6.152742,0.050131,131
14962,P. Manca,1342167297,FSU,FA,L,84,90.796429,22.894048,-9.064286,2388.154762,-1.719048,6.700000,5.780952,0.051013,131
17675,S. Sandford,1464054215,FLA,FA,R,222,95.184234,17.827027,10.715766,2306.490991,3.327928,5.901802,6.308559,0.052770,130
13188,C. West,1310862081,CONN,FA,L,529,90.734405,23.150662,-5.896597,2241.047259,-0.365217,6.194140,6.011531,0.052814,130
19247,J. Nottingham,1557710177,UGA,FA,R,156,95.980769,21.044872,12.182692,2556.910256,1.534615,6.179487,5.990385,0.052677,130


In [48]:
stuff[stuff["PitchType"] == "SL"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
3598,A. Buczkowski,1123936875,CIN,SL,R,332,81.022590,10.767169,-12.430723,2579.635542,5.368976,3.634639,5.293976,0.020201,146
12364,H. Dietz,1301090561,ARK,SL,L,129,85.413178,1.497674,1.931783,2494.410853,-2.406977,6.617829,5.520155,0.036062,133
14018,C. Benge,1330701056,LSU,SL,R,32,84.621875,6.696875,-13.293750,2490.781250,2.043750,5.140625,4.928125,0.037845,132
4772,C. Stokes,1152980224,FSU,SL,R,162,84.927160,5.427778,-14.377160,2550.419753,1.790741,5.505556,5.485185,0.039366,131
13618,M. Yehl,1319280896,WVU,SL,L,253,84.630040,1.056522,10.787352,2627.106719,-3.085375,5.922925,5.613834,0.039241,131
22783,K. Graham,1753862210,USA,SL,R,36,82.775000,8.766667,-8.566667,2427.472222,3.113889,5.677778,6.052778,0.040243,130
9973,J. Flora,1254299136,UCSB,SL,R,218,85.425229,6.351835,-10.991743,2606.756881,2.370183,5.574771,5.882569,0.042973,128
669,M. Brassfield,1038792681,TCU,SL,L,383,84.926893,-0.371802,7.160313,2746.587467,-1.909138,6.244909,5.461358,0.043042,128
1436,J. Dimino,1078910874,UNF,SL,L,72,83.198611,-0.627778,11.422222,2619.472222,-1.319444,6.200000,5.677778,0.043798,127
3810,D. Whitney,1128102459,ORST,SL,R,114,86.636842,-0.489474,-11.597368,2618.517544,1.560526,6.338596,5.850000,0.043835,127


In [49]:
stuff[stuff["PitchType"] == "CH"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
12743,B. Purcell,1307080960,FSU,CH,R,93,85.370968,4.695699,15.644086,1378.559140,0.355914,6.738710,5.109677,0.042274,138
7816,D. Jackson,1205651242,UCSB,CH,L,148,76.597973,3.318919,-20.152027,2558.668919,-1.581081,5.831081,5.795270,0.042210,138
21688,S. Shenosky,1689800810,PSU,CH,R,28,75.114286,8.253571,16.017857,2057.857143,1.239286,6.085714,7.164286,0.050256,132
10225,S. McCabe,1258728572,COFC,CH,R,25,77.088000,8.768000,20.328000,2339.240000,1.544000,5.772000,5.400000,0.053585,130
12040,G. Naess,1296842752,CP,CH,R,274,75.827007,7.848905,19.596715,2103.237226,1.565328,6.447810,5.786861,0.053207,130
25896,C. Sova,1939270667,BGSU,CH,R,88,77.178409,4.101136,21.100000,2193.556818,1.626136,5.786364,6.242045,0.053892,129
8204,R. Featherston,1212946688,ORE,CH,R,72,83.895833,-1.580556,13.138889,1004.777778,0.700000,5.681944,6.162500,0.056685,127
6905,J. Mitchell,1194506240,BING,CH,R,135,84.869630,8.041481,14.157778,1371.540741,0.082963,6.668889,5.860741,0.056414,127
19727,C. Chatterton,1584179280,AUB,CH,R,114,75.914912,8.771053,14.382456,2329.561404,0.625439,5.903509,5.513158,0.057196,127
23222,C. Hartsell,1778925046,BC,CH,R,98,80.470408,8.169388,21.462245,2280.510204,1.359184,5.521429,6.070408,0.058701,126


In [51]:
stuff[stuff["PitchType"] == "CU"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26547,D. Volantis,1979617275,TEX,CU,L,383,81.376762,-12.064752,3.326110,2586.378590,-2.091384,6.800522,5.224021,0.004106,153
12360,H. Dietz,1301090561,ARK,CU,L,136,80.165441,-14.566912,7.788235,2741.213235,-2.185294,6.729412,5.277941,0.018632,142
21330,B. Trosclair,1674977859,NWST,CU,L,29,78.096552,-3.244828,10.444828,2977.862069,-2.155172,5.551724,6.468966,0.023093,139
9469,H. Leffew,1239346432,TEX,CU,L,136,80.859559,-11.393382,0.289706,2407.654412,-0.688235,6.745588,5.331618,0.038788,127
11007,K. English,1275287609,IND,CU,R,27,83.503704,-3.477778,-2.562963,2433.666667,2.051852,6.988889,5.292593,0.040644,126
3732,J. Gomberg,1126277376,UNLV,CU,L,27,84.370370,-9.518519,6.170370,2637.259259,-1.444444,5.381481,4.992593,0.040463,126
16520,M. Stevenson,1386657229,BEL,CU,R,59,76.408475,-9.955932,-17.988136,2923.745763,3.325424,5.954237,5.391525,0.042085,125
12521,M. Knight,1304692992,MOSU,CU,L,309,77.913916,-12.184790,16.516828,2985.928803,-1.809061,6.186084,5.540453,0.044103,123
16027,C. Townsend,1357477972,MISS,CU,R,124,82.676613,-14.775000,-3.700000,2567.024194,1.506452,6.269355,5.483871,0.044791,123
22572,C. Foster,1740630964,MSST,CU,L,58,78.265517,-13.189655,5.348276,2530.137931,-1.415517,6.563793,5.417241,0.044775,123


In [52]:
stuff[stuff["PitchType"] == "FC"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
5097,G. Helsel,1162012672,XAV,FC,L,181,78.374033,7.903867,7.519890,2188.348066,-2.481768,6.627072,5.295028,-0.002987,151
15884,H. Han,1348178401,TTU,FC,R,30,79.636667,0.363333,-3.690000,2503.666667,2.513333,3.533333,5.043333,0.019946,137
22429,J. Lucas,1734064400,TROY,FC,L,29,77.310345,5.875862,10.631034,2427.344828,-2.465517,5.265517,5.855172,0.021360,136
13459,M. Paz,1315747265,LSU,FC,R,30,89.346667,2.633333,-0.683333,2698.466667,2.910000,6.490000,5.006667,0.024604,134
11590,D. Pressley,1285478656,WEBB,FC,R,32,82.721875,6.568750,0.575000,2287.812500,0.875000,7.028125,5.084375,0.026926,133
15410,A. Vargas,1342169018,NICH,FC,L,164,87.713415,6.831098,1.590854,2308.957317,-1.727439,6.773171,6.456707,0.028723,132
12362,H. Dietz,1301090561,ARK,FC,L,364,85.184890,1.099451,1.905495,2472.524725,-2.402473,6.522253,5.571703,0.029314,132
5342,R. White,1162368768,WCU,FC,L,28,79.221429,2.992857,3.771429,2437.928571,-2.678571,4.253571,5.017857,0.035108,128
21862,W. Jordan,1698188238,TTU,FC,R,64,80.595313,0.517188,-0.792188,2239.406250,1.678125,6.750000,5.195312,0.036725,127
12143,I. Williams,1297495552,OKLA,FC,R,42,89.304762,8.600000,-3.892857,2644.857143,2.607143,5.461905,5.271429,0.037273,127


In [53]:
stuff[stuff["PitchType"] == "SI"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
168,D. Hamilton,1008376863,CARK,SI,R,62,90.466129,17.624194,17.533871,2455.758065,2.495161,5.132258,5.620968,0.048104,140
1170,M. Harsch,1065101304,WAKE,SI,R,65,96.141538,7.783077,15.910769,1816.892308,0.967692,5.933846,5.929231,0.049210,139
10385,R. Hughes,1262528256,TOL,SI,R,42,92.028571,14.850000,12.652381,2022.738095,1.614286,7.754762,5.790476,0.050860,138
26453,C. Van Ameyde,1970680555,ND,SI,R,188,91.592553,8.014362,21.623936,2321.840426,3.339894,3.878723,6.241489,0.053605,136
14326,E. Norby,1335271936,ECU,SI,L,42,91.302381,11.959524,-13.276190,2551.476190,-2.890476,5.130952,6.359524,0.057117,134
20291,N. Romer,1609880601,SJU,SI,R,63,81.647619,4.082540,23.595238,2070.126984,3.503175,3.582540,6.000000,0.058865,133
7554,K. Hawks,1201847438,MORE,SI,L,32,89.421875,20.403125,-15.962500,2149.187500,-2.125000,6.303125,5.865625,0.064011,130
22281,C. Appenzeller,1722877754,TENN,SI,L,81,90.393827,13.477778,-18.138272,2244.407407,-2.291358,5.479012,6.238272,0.062985,130
7581,M. Gosztola,1202396160,ORE,SI,L,62,91.604839,12.298387,-19.624194,2068.112903,-2.391935,5.470968,6.175806,0.065805,129
18356,N. Toups,1505713358,SELA,SI,L,308,92.798052,11.516234,-18.668506,2207.337662,-1.937013,5.463961,6.018831,0.066955,128


In [54]:
stuff[stuff["PitchType"] == "FS"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
4370,J. Noot,1142711296,LSU,FS,R,44,87.709091,-0.156818,16.836364,1514.000000,1.906818,6.159091,4.927273,0.029304,121
14300,J. Giordano,1334554354,RICH,FS,L,92,76.967391,6.582609,-11.223913,1541.282609,-1.542391,5.976087,5.032609,0.031568,121
5046,L. Earnhardt,1160764160,WIN,FS,L,87,78.150575,2.075862,-6.420690,1068.712644,-1.281609,6.490805,5.294253,0.040473,118
3494,L. Kenny,1120804043,UOWG,FS,R,74,82.048649,-0.036486,13.394595,997.270270,1.036486,6.282432,5.335135,0.042172,117
19257,K. Sweum,1558099259,GONZ,FS,L,42,81.533333,4.959524,-3.647619,1069.809524,-1.447619,6.254762,5.445238,0.049675,115
23440,P. Bauer,1791552299,JMU,FS,R,36,79.286111,1.275000,11.566667,1032.500000,2.005556,6.544444,6.144444,0.048488,115
19769,H. Bond,1586062407,TXAM,FS,L,60,80.478333,8.681667,-13.788333,1497.583333,-2.618333,6.335000,5.670000,0.051368,114
1019,D. Brousseau,1055997641,BYU,FS,R,54,81.464815,-1.431481,9.312963,1141.629630,2.027778,6.033333,5.981481,0.053456,114
7793,O. Marsh,1205184939,BEL,FS,R,53,82.077358,5.373585,6.828302,1676.283019,0.983019,6.083019,5.596226,0.052775,114
6806,B. Fridgen,1191930494,PENN,FS,R,25,80.424000,2.556000,16.876000,1688.120000,2.312000,4.760000,5.224000,0.056079,113


In [55]:
# Save stuff df as a csv
stuff.to_csv("stuff.csv")

In [56]:
stuff[stuff["Count"] > 200].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26547,D. Volantis,1979617275,TEX,CU,L,383,81.376762,-12.064752,3.326110,2586.378590,-2.091384,6.800522,5.224021,0.004106,153
3598,A. Buczkowski,1123936875,CIN,SL,R,332,81.022590,10.767169,-12.430723,2579.635542,5.368976,3.634639,5.293976,0.020201,146
5073,J. Music,1161348114,CAMP,FA,R,212,92.099528,21.347170,13.038208,2209.424528,3.513208,5.965566,5.712736,0.047375,133
4975,B. Frederick,1158279680,TENN,FA,R,319,82.363636,1.687461,7.452351,2176.504702,2.923197,1.651724,5.718182,0.047789,132
12362,H. Dietz,1301090561,ARK,FC,L,364,85.184890,1.099451,1.905495,2472.524725,-2.402473,6.522253,5.571703,0.029314,132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15579,T. Bourque,1342169568,WM,FC,L,201,84.836816,9.525871,-2.108458,2310.572139,-0.662189,6.272637,5.847761,0.124215,75
2940,L. Nichols,1104573440,LT,SI,L,320,88.250625,5.675625,-13.345625,1906.518750,-1.376562,5.210312,6.527500,0.151974,75
22909,L. Doll,1760080732,MRSH,CU,L,225,72.558667,-7.759111,11.696000,2126.328889,-2.870667,5.102222,6.429778,0.112217,72
5176,E. Sargent,1162067200,UMBC,SL,L,314,75.837898,-0.730573,8.186624,2231.980892,-1.948089,4.191401,5.413376,0.113997,69


In [58]:
stuff[(stuff["Hand"] == "R") & (stuff["PitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
5073,J. Music,1161348114,CAMP,FA,R,212,92.099528,21.347170,13.038208,2209.424528,3.513208,5.965566,5.712736,0.047375,133
4975,B. Frederick,1158279680,TENN,FA,R,319,82.363636,1.687461,7.452351,2176.504702,2.923197,1.651724,5.718182,0.047789,132
24696,J. Barberi,1862463014,FLA,FA,R,260,97.502308,20.915385,10.915385,2533.519231,1.840769,6.333077,5.432308,0.051253,131
26198,C. Clark,1956935142,USM,FA,R,428,93.192290,21.430841,9.149065,2264.841121,1.804206,5.489953,6.488084,0.052906,130
17675,S. Sandford,1464054215,FLA,FA,R,222,95.184234,17.827027,10.715766,2306.490991,3.327928,5.901802,6.308559,0.052770,130
19247,J. Nottingham,1557710177,UGA,FA,R,156,95.980769,21.044872,12.182692,2556.910256,1.534615,6.179487,5.990385,0.052677,130
4230,J. Raab,1139725824,GTWN,FA,R,252,92.604762,22.755952,11.202381,2514.500000,2.103571,6.135317,6.111905,0.053812,129
12745,B. Purcell,1307080960,FSU,FA,R,279,92.278136,23.750179,8.527240,2330.175627,0.165950,6.885305,5.100358,0.055432,128
3375,C. Howard,1116162816,TEX,FA,R,97,94.542268,16.541237,8.734021,2163.329897,3.606186,4.978351,6.578351,0.055600,128
21716,L. ODonnell,1691990270,USF,FA,R,98,95.098980,21.580612,9.894898,2475.030612,1.166327,6.293878,6.212245,0.058531,127


In [21]:
import pandas as pd
import numpy as np
stuff = pd.read_csv("stuff.csv")
pitches = list(stuff["PitchType"].unique())

In [59]:
# Converting stuff csv to a leaderboard format, where each pitcher/pitch type is in one row

rows = []
ids = list(stuff["pitcherId"].unique())

for i in ids:
    x = stuff[stuff["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["team"] = x["team"].iloc[0]
    dx["Hand"] = x["Hand"].iloc[0]
    for pt in list(stuff["PitchType"].unique()):
        if pt in list(x["PitchType"].unique()):
            dx["Stf+ " + pt] = x[x["PitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["PitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)

In [60]:
table = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table.columns if col.startswith("Stf+")]
table[stf_cols] = table[stf_cols].astype("Int64")

# Calculating overall Stuff+ based on weighted average of each Stuff+ value. More pitch usage = more weight
table["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table.columns and ct_col in table.columns:
        mask = table[col].notna()
        table.loc[mask, "Stuff+"] += (
            table.loc[mask, col] * (table.loc[mask, ct_col] / table.loc[mask, "Total"])
        )

table["Stuff+"] = table["Stuff+"].round().astype(int)
table = table[["Pitcher", "pitcherId", "team", "Total", "Hand", "Stf+ FA", "Stf+ SL", "Stf+ CU", "Stf+ FC", "Stf+ SI", "Stf+ CH", "Stf+ FS", "Stuff+"]]

In [61]:
table[table["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
3201,M. Santos,1487988943,URI,390,R,113,111,<NA>,<NA>,<NA>,<NA>,<NA>,112
2205,P. Aikens,1299756340,URI,355,L,97,123,<NA>,<NA>,<NA>,<NA>,<NA>,110
2858,J. Cullen,1358089441,URI,726,R,119,107,106,104,<NA>,94,<NA>,110
1369,C. Grotyohann,1203468032,URI,296,R,116,101,<NA>,98,<NA>,<NA>,<NA>,109
513,A. Jones,1101042688,URI,262,R,110,104,109,103,<NA>,<NA>,<NA>,108
2564,C. Maloney,1336247552,URI,175,R,110,105,<NA>,<NA>,<NA>,<NA>,<NA>,108
2198,W. Creed,1299086745,URI,68,L,107,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,107
579,J. Sabbath,1110853376,URI,768,R,106,110,99,<NA>,<NA>,96,<NA>,106
2733,B. Perry,1342168540,URI,56,R,104,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,104
1227,S. Houchens,1194448803,URI,606,L,105,101,<NA>,<NA>,<NA>,103,<NA>,103


In [62]:
table[table["Total"] >= 500].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
2283,B. Purcell,1307080960,FSU,512,R,128,113,<NA>,<NA>,<NA>,138,<NA>,126
1287,R. Marohn,1197348608,NCST,638,L,132,<NA>,107,<NA>,<NA>,122,<NA>,126
4706,D. Volantis,1979617275,TEX,1018,L,102,116,153,120,103,84,<NA>,125
2217,H. Dietz,1301090561,ARK,1195,L,115,133,142,132,<NA>,<NA>,<NA>,125
905,G. Helsel,1162012672,XAV,632,L,95,126,<NA>,151,<NA>,110,<NA>,125
4377,J. Barberi,1862463014,FLA,503,R,131,117,<NA>,<NA>,<NA>,<NA>,<NA>,124
1296,S. Matson,1198395392,USC,580,L,131,104,<NA>,<NA>,<NA>,<NA>,<NA>,122
780,L. Peterson,1142994176,FLA,1299,R,125,127,112,120,<NA>,103,<NA>,121
2024,M. Edwards,1279382784,USC,1264,L,124,113,120,<NA>,<NA>,113,<NA>,121
3301,W. Sanford,1518897568,ORE,1244,R,126,101,116,<NA>,<NA>,109,<NA>,121


In [63]:
table[table["Total"] >= 100].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
4688,C. Van Ameyde,1970680555,ND,188,R,<NA>,<NA>,<NA>,<NA>,136,<NA>,<NA>,136
3415,J. Nottingham,1557710177,UGA,156,R,130,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,130
127,C. Powell,1041792638,TXAM,125,L,129,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,129
2668,P. Manca,1342167297,FSU,109,L,131,116,<NA>,<NA>,<NA>,<NA>,<NA>,128
1287,R. Marohn,1197348608,NCST,638,L,132,<NA>,107,<NA>,<NA>,122,<NA>,126
2283,B. Purcell,1307080960,FSU,512,R,128,113,<NA>,<NA>,<NA>,138,<NA>,126
4061,N. Helman,1756665469,KENN,420,R,125,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,125
2217,H. Dietz,1301090561,ARK,1195,L,115,133,142,132,<NA>,<NA>,<NA>,125
3140,S. Sandford,1464054215,FLA,312,R,130,112,<NA>,<NA>,<NA>,<NA>,<NA>,125
4706,D. Volantis,1979617275,TEX,1018,L,102,116,153,120,103,84,<NA>,125


In [64]:
table[table["Total"] >= 100].sort_values("Stf+ FS", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
776,J. Noot,1142711296,LSU,124,R,95,<NA>,<NA>,<NA>,108,<NA>,121,109
2554,J. Giordano,1334554354,RICH,916,L,106,104,98,<NA>,<NA>,103,121,106
896,L. Earnhardt,1160764160,WIN,871,L,102,100,105,<NA>,<NA>,103,118,104
626,L. Kenny,1120804043,UOWG,693,R,100,104,<NA>,107,<NA>,111,117,105
4166,P. Bauer,1791552299,JMU,771,R,95,103,79,<NA>,<NA>,<NA>,115,96
3417,K. Sweum,1558099259,GONZ,1029,L,112,124,<NA>,100,<NA>,100,115,113
1389,O. Marsh,1205184939,BEL,328,R,95,102,<NA>,<NA>,<NA>,<NA>,114,101
182,D. Brousseau,1055997641,BYU,429,R,96,<NA>,<NA>,<NA>,98,113,114,101
3024,P. Green,1424021277,MIZZ,198,R,109,109,<NA>,<NA>,<NA>,<NA>,111,109
1271,C. Estridge,1196802304,WVU,404,R,100,101,<NA>,<NA>,102,<NA>,111,103


In [66]:
# Save Stuff+ leaderboard as csv file
table.to_csv("stuff_table.csv")